# Apache HBase

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/hbase/hbase.ipynb)

Apache HBase is a distributed, scalable, wide-column NoSQL store modeled on Google Bigtable and built on top of HDFS. It gives random, real-time reads and writes by row key over tables with billions of rows and millions of columns, while HDFS provides the durable storage underneath.

The runnable cells simulate the mechanisms in plain Python (row-key routing to regions, the versioned cell map, delete markers, TTL, filters); shell, Java, Phoenix, and client snippets are fenced.

```text
client -> ZooKeeper (who is the master, where is hbase:meta)
       -> RegionServer -> region -> MemStore + WAL + HFiles on HDFS
```

Key ideas:

+ Rows are sorted lexicographically by row key and split into regions; a region is the unit of distribution, not a table.
+ A column family is the physical storage and tuning unit; column qualifiers inside it are fully dynamic.
+ Writes land in the write-ahead log (WAL) and the in-memory MemStore, and are flushed to immutable HFiles later.
+ Reads check the MemStore and block cache, then the HFiles; tombstones hide deletes until compaction removes them.
+ HBase is optimized for key-based access and scans, not for joins, ad-hoc SQL analytics, or multi-row transactions across tables.

## How do you design an HBase row key?

An HBase row key should support the main lookup pattern and avoid region hotspotting. Because rows are sorted by key and a region serves a contiguous key range, monotonically increasing keys (timestamps, sequential ids) overload the last region. Use salting, hashing, bucketing, or reversed timestamps to spread writes.

| Access pattern | Row key design |
|---|---|
| latest events for a user | `user_id + reverse_timestamp` |
| devices by tenant | `salt + tenant_id + device_id` |
| time-range scans, spread writes | `salt + reverse_timestamp + id` |
| lookup by many secondary keys | separate index/lookup table keyed by the secondary key |
| natural unique id | hash(id) or a fixed-width, zero-padded id |

```text
raw row key:        device-0001|2026-10-04T13:20:00Z
salted row key:     07|device-0001|2026-10-04T13:20:00Z   (salt = hash(device_id) % 16)
bucketed row key:   device-0001|bkt-3|1700000000000
reversed timestamp: device-0001|9999999999999-1700000000000
```

**Mechanics.** HBase routes by key range, not by row identity. `hbase:meta` is itself an HBase table whose row keys are `table,startKey,regionId`; the client scans it once, caches the region boundaries, and binary-searches them for every request. The row key therefore decides both the sort order inside a region and which RegionServer answers. A key that increases forever (a raw timestamp, a sequential id) always sorts past every other key, so it always lands in the last region and one machine absorbs every write.

**Worked example.** A raw event key `device-0001|2026-10-04T13:20:00Z` puts all traffic for one device in one region. Rewritten as `07|device-0001|9999999999999-1700000000000`, the device moves into bucket 7 of 16; the bucket prefixes `00` to `0f` are spread across the table's regions, and the reversed timestamp sorts the newest event first inside the device.

**Common mistake.** Salting with two or four buckets only relocates the hotspot. Choose the bucket count so `expected rows / buckets` is close to the target region size, and format the salt at fixed width (`%02d`) so lexicographic order matches numeric order. Salting also fans out reads: a query for one device must know its bucket, or it scans all buckets and multiplies read load.

In [ ]:
# Build a salted, reverse-timestamp row key for recent-first lookups
import hashlib

def rowkey(device_id: str, ts_ms: int, buckets: int = 16) -> str:
    salt = int(hashlib.md5(device_id.encode()).hexdigest(), 16) % buckets
    reverse_ts = 9_999_999_999_999 - ts_ms
    return f"{salt:02d}|{device_id}|{reverse_ts:013d}"


# Route keys to 16 pre-split regions ("00|" .. "15|") and compare write spread.
import bisect
from collections import Counter

SPLITS = [f"{n:02d}|" for n in range(1, 16)]          # region i holds keys in [SPLITS[i-1], SPLITS[i])


def region_for(row_key: str) -> int:
    return bisect.bisect_right(SPLITS, row_key)


base_ms = 1_791_000_000_000
devices = [f"device-{n:04d}" for n in range(2000)]
writes = [(d, base_ms + s * 1000) for s in range(5) for d in devices]

time_first = Counter(region_for(f"{ts:013d}|{d}") for d, ts in writes)
salted = Counter(region_for(rowkey(d, ts)) for d, ts in writes)
assert len(time_first) == 1                           # every write lands in the last region
assert len(salted) == 16 and max(salted.values()) < 2 * min(salted.values())

recent = sorted(rowkey("device-0007", base_ms + s * 1000) for s in range(3))
assert recent[0] == rowkey("device-0007", base_ms + 2000)   # reverse timestamp: newest sorts first
print("regions hit, timestamp-first key:", len(time_first), "| salted key:", len(salted))
print("newest key for device-0007:", recent[0])

The trade-off: reads for one device must query every salt bucket (or you scan all prefixes) unless the salt is derivable from the query. Salting spreads writes but fans out reads; reverse timestamps give newest-first scans for a known prefix.

Hotspotting is when a large amount of client traffic is directed at one node, or a few nodes, of a cluster. The traffic can be reads, writes, or other operations, and it overwhelms the machine hosting that region, causing performance degradation and potentially region unavailability. The classic cause is a monotonically increasing row key such as a timestamp or sequential id, which always writes to the last region.

Fixes, in the order an interviewer usually wants to hear them:

+ Salt the row key with a hash prefix so writes spread across regions.
+ Use a reverse timestamp so the newest row is first for a known prefix but writes are not sequential across users.
+ Pre-split the table so the key space starts balanced.
+ Use a random or hashed key when point lookups are by a secondary key.

Salting costs read fan-out: a query for one user must know the salt (derive it from the user id) rather than scanning all buckets.

## When should you use HBase?

Use HBase for sparse, wide, high-scale tables that need random reads/writes by key over large datasets. It fits operational lookups, time-series style access, and serving stores behind an application. It is not ideal for ad-hoc SQL analytics, joins, or full table scans.

+ good: get the latest 100 events for a device id
+ poor: calculate total revenue by region across all history

Analytics usually belongs in Hive, Spark, Impala, or a warehouse, while HBase serves key-based access. A common pattern is to serve from HBase and run analytics on the same data copied to Parquet or Iceberg.

+ Analytical SQL over the whole dataset - use a warehouse (Snowflake, BigQuery, Redshift) or a lakehouse table (Iceberg, Delta, Hudi) on Parquet. HBase scans are not columnar and do not support joins.
+ Low-latency key-value at very high write rates with multi-region writes - consider Cassandra or Bigtable, which offer multi-datacenter replication without the RegionServer/ZooKeeper stack.
+ Ordered streaming with replay and many consumers - use Kafka as the backbone, not HBase.
+ Small or medium datasets that fit in a relational database - use PostgreSQL or MySQL; HBase only pays off at scale.
+ Mostly write-once time series analytics - use Parquet/Iceberg with time partitioning; HBase is for point access, not aggregation.
+ Strong multi-row ACID transactions - use a relational or NewSQL database, not HBase.

A useful rule: HBase serves a known key to an application. If the question is "what does this key hold right now", HBase is a candidate; if the question is "aggregate everything matching a predicate", it is not.

## What is the HBase data model?

HBase comprises a set of tables. Each table has rows and column families, and the row key acts as the primary key used for every access. A cell is addressed by row key, column family, qualifier, and timestamp, and each cell can hold multiple versions. Column qualifiers are dynamic per row, so a table can be extremely sparse and wide.

```text
table
  row key (sorted) -> column family -> qualifier -> [version, value]
```

**Mechanics.** The model is a four-level sorted map: table, row key, column family plus qualifier, timestamped value. Row keys are sorted byte-wise and split into regions; families are declared at create time and are the storage unit; qualifiers are created implicitly on first write. A cell holds the value bytes plus a type byte, and versions are ordered newest first.

```text
table -> row key (sorted) -> family -> qualifier -> [(timestamp, value)]
```

**Worked example.** One row can hold two families with different version policies: `i:amount` keeps one version for 90 days, `meta:note` keeps three versions with no TTL. Both live in the same row but in separate stores with separate HFiles.

**Common mistake.** Expecting a schema registry or types. HBase stores bytes, so a `put` of the string `"42.10"` and a `put` of the double `42.1` are different byte arrays. Enforce types in the application or let Phoenix declare them, and be consistent about byte order and encoding across writers.

A column family is the basic unit of physical storage and the level at which storage features such as compression, block size, Bloom filters, versions, and TTL are configured. All cells in a family for a row are stored together on the same RegionServer, so families should be defined by access pattern, not by convenience.

Keep the number of column families small (typically 1-3): each family has its own MemStore and flush behavior, and many families multiply memory pressure and flush storms.

```bash
# One family for the hot lookup path, one for cold metadata
create 'app:impressions', {NAME => 'i', VERSIONS => 1}, {NAME => 'meta', VERSIONS => 3}
```

A row key identifies a row within a table, not within a column family. All column families in that row share the same row key; a row can exist with cells in only some (or none explicitly, if a family marker is written) of the families. Two different column families do not have separate row key spaces.

+ Tall-and-thin: many rows, few columns per row. This is the default HBase sweet spot and scales best because rows spread across regions.
+ Short-and-wide: fewer rows, many columns per row. Useful when a natural entity is read as a whole and the number of entities is bounded, but a single row cannot be split across regions, so it limits write parallelism.

Note the common confusion: "short and wide" means few rows and many columns, not few columns and many rows.

The model as code: a sorted map of `row -> family:qualifier -> {timestamp: value}`, with `VERSIONS`, TTL, the three delete markers, and scans that walk row keys in order.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Family:
    versions: int = 1
    ttl_s: int | None = None


@dataclass
class HTable:
    families: dict[str, Family]
    rows: dict[str, dict[str, dict[int, str]]] = field(default_factory=dict)   # row -> col -> {ts: value}
    markers: list[tuple] = field(default_factory=list)                          # (kind, row, col_or_family, ts)

    def put(self, row: str, column: str, value: str, ts: int) -> None:
        assert column.split(":")[0] in self.families, "family must exist; qualifiers are free-form"
        self.rows.setdefault(row, {}).setdefault(column, {})[ts] = value

    def delete(self, kind: str, row: str, target: str, ts: int) -> None:
        self.markers.append((kind, row, target, ts))   # a delete is a write: a tombstone marker

    def _masked(self, row: str, column: str, ts: int) -> bool:
        family = column.split(":")[0]
        for kind, r, target, mts in self.markers:
            if r != row:
                continue
            if kind == "version" and target == column and ts == mts:
                return True
            if kind == "column" and target == column and ts <= mts:
                return True
            if kind == "family" and target == family and ts <= mts:
                return True
        return False

    def get(self, row: str, now: int, max_versions: int = 1) -> dict[str, list[tuple[int, str]]]:
        result = {}
        for column, cells in self.rows.get(row, {}).items():
            fam = self.families[column.split(":")[0]]
            live = [(ts, v) for ts, v in sorted(cells.items(), reverse=True)
                    if not self._masked(row, column, ts)
                    and (fam.ttl_s is None or now - ts < fam.ttl_s * 1000)][:min(max_versions, fam.versions)]
            if live:
                result[column] = live
        return result

    def scan(self, start: str = "", stop: str = "\uffff", now: int = 0, row_filter=None, limit=None):
        out = []
        for row in sorted(self.rows):                  # rows are stored sorted by key
            if start <= row < stop:
                cells = self.get(row, now)
                if cells and (row_filter is None or row_filter(row, cells)):
                    out.append((row, {c: v[0][1] for c, v in cells.items()}))
            if limit and len(out) == limit:
                break
        return out


t = HTable({"i": Family(versions=3, ttl_s=90 * 86400), "m": Family()})
t.put("u42|a1", "i:campaign_id", "cmp-98", ts=1_000)
t.put("u42|a1", "i:campaign_id", "cmp-99", ts=2_000)   # new version, same cell
t.put("u42|a1", "i:amount", "42.10", ts=2_000)
t.put("u42|a1", "m:ua", "firefox", ts=2_000)
t.put("u43|a7", "i:campaign_id", "cmp-99", ts=2_500)

now = 3_000
assert t.get("u42|a1", now)["i:campaign_id"] == [(2_000, "cmp-99")]
assert len(t.get("u42|a1", now, max_versions=5)["i:campaign_id"]) == 2

t.delete("version", "u42|a1", "i:campaign_id", ts=2_000)   # removes one version
assert t.get("u42|a1", now)["i:campaign_id"] == [(1_000, "cmp-98")]
t.delete("family", "u42|a1", "m", ts=2_000)                # removes the whole family up to ts
assert "m:ua" not in t.get("u42|a1", now)
t.put("u42|a1", "m:ua", "chrome", ts=1_500)                # older timestamp than the marker
assert "m:ua" not in t.get("u42|a1", now)                  # masked until a major compaction purges it

assert t.get("u42|a1", now=91 * 86_400_000) == {}          # past the 90-day TTL: expired on read
prefix = t.scan(start="u42|", stop="u42}", now=now)        # "}" sorts right after "|"
assert [row for row, _ in prefix] == ["u42|a1"]
print(t.get("u42|a1", now, max_versions=3))

## What are the key components of HBase?

+ ZooKeeper: coordinates the cluster, holds the `hbase:meta` location and master election.
+ HBase Master (HMaster): assigns regions, balances load, and performs admin operations.
+ RegionServer: serves regions and handles reads, writes, and flushes.
+ Region: a contiguous range of a table's rows, containing a MemStore and HFiles.
+ Catalog tables: `hbase:meta` maps regions to RegionServers (the older `-ROOT-` table is gone in modern HBase).
+ HDFS: durable storage for WALs and HFiles.

**Mechanics.** The components form a strict hierarchy. ZooKeeper holds the `hbase:meta` location and elects the active master. The HMaster assigns regions and balances load but is off the data path. Each RegionServer hosts regions, appends to the WAL, keeps MemStores, and flushes HFiles to HDFS. `hbase:meta` is the catalogue that maps a key range to the region and server that owns it.

```text
ZooKeeper -> HMaster (assignment, DDL, balancing)
          -> hbase:meta (key range -> region -> RegionServer)
          -> RegionServer -> region -> store (one per family) -> MemStore + HFiles on HDFS
```

**Common mistake.** Believing the client asks the master where data lives. It does not: clients cache region locations from `hbase:meta` and talk to RegionServers directly, which is why a master restart does not interrupt serving and why a stale client cache shows up as `RegionMovedException` retries.

### What do the HMaster, RegionServer, ZooKeeper, and hbase:meta do?

The Master server assigns regions to RegionServers and handles load balancing across the cluster. It also performs admin operations such as create, alter, and drop table, and coordinates region splits, merges, and recovery. It is not on the read/write data path: clients talk to RegionServers directly, so a brief master outage does not stop serving.

A table is divided into regions, and a RegionServer serves a set of regions to clients. It handles reads and writes, keeps the MemStore, appends to the WAL, flushes to HFiles, and runs compactions and splits. In a distributed cluster a RegionServer runs on a DataNode.

ZooKeeper maintains cluster configuration and the communication path between RegionServers and clients. It also provides distributed synchronization and holds cluster state through sessions. Every RegionServer and the HMaster send heartbeats to ZooKeeper, which detects failures and provides server-failure notifications so recovery can start. Clients also use ZooKeeper to find `hbase:meta`.

Catalog tables maintain metadata about the cluster. `hbase:meta` maps each region to its RegionServer and start row key. In modern HBase the old `-ROOT-` table no longer exists; the location of `hbase:meta` is stored in ZooKeeper, and clients cache region locations locally.

Yes. In a normal deployment RegionServers run on the same hosts as DataNodes so that HDFS data locality lets a RegionServer read its regions from local disks.

## How do you create and query tables in the HBase shell?

The HBase shell is a JRuby REPL that talks to the cluster for DDL and DML. Namespaces, tables, and column families are created here; `disable` is required before changing a table.

```bash
# Namespace and table DDL: TTL, versions, block size, and Bloom filter per column family
create_namespace 'app'
create 'app:impressions', {NAME => 'i', VERSIONS => 1, TTL => 7776000, BLOOMFILTER => 'ROWCOL', BLOCKSIZE => 65536}
describe 'app:impressions'

# Add a column family to an existing table
disable 'app:impressions'
alter 'app:impressions', {NAME => 'meta', VERSIONS => 3}
enable 'app:impressions'
```

```bash
# DML: put, get, scan, increment, delete, count
put 'app:impressions', 'u42|9999999999999-1700000000000|a1', 'i:campaign_id', 'cmp-99'
put 'app:impressions', 'u42|9999999999999-1700000000000|a1', 'i:amount', '42.10'
get 'app:impressions', 'u42|9999999999999-1700000000000|a1'
scan 'app:impressions', {STARTROW => 'u42|', ENDROW => 'u42|\xff', LIMIT => 5}
incr 'app:counters', 'daily|2026-10-04', 'c:impressions', 1
delete 'app:impressions', 'u42|9999999999999-1700000000000|a1', 'i:amount'
count 'app:impressions'
```

```bash
# Table maintenance and reporting
status 'detailed'
list_regions 'app:impressions'
flush 'app:impressions'
major_compact 'app:impressions'
```

+ `put` - writes a cell value at a specified row and column.
+ `get` - fetches a row or a specific cell.
+ `delete` - deletes a cell value.
+ `deleteall` - deletes all cells in a row.
+ `scan` - scans and returns table data.
+ `count` - counts the rows in a table.
+ `truncate` - disables, drops, and recreates a table.
+ `incr` - atomically increments a counter cell.
+ `append` - appends bytes to an existing cell value.

A namespace is a logical grouping of tables, similar to a database or schema in a relational database. It organizes tables and controls quotas and permissions. The default namespaces are `default` and `hbase`.

```bash
create_namespace 'app'
create 'app:impressions', 'i'
```

## How do you configure a cluster with hbase-site.xml?

`hbase-site.xml` holds cluster-wide settings; the most important ones control durability, write throughput, and MemStore sizing. The pattern is one `property` block per setting.

```xml
<configuration>
  <property>
    <name>hbase.rootdir</name>
    <value>hdfs://nn.example.com:8020/hbase</value>
  </property>
  <property>
    <name>hbase.cluster.distributed</name>
    <value>true</value>
  </property>
  <property>
    <name>hbase.zookeeper.quorum</name>
    <value>zk1.example.com,zk2.example.com,zk3.example.com</value>
  </property>
  <property>
    <name>hbase.regionserver.handler.count</name>
    <value>60</value>
  </property>
  <property>
    <name>hbase.regionserver.global.memstore.size</name>
    <value>0.4</value>
  </property>
  <property>
    <name>hbase.hregion.memstore.flush.size</name>
    <value>134217728</value>
  </property>
  <property>
    <name>hbase.hregion.max.filesize</name>
    <value>10737418240</value>
  </property>
  <property>
    <name>hbase.hstore.blockingStoreFiles</name>
    <value>16</value>
  </property>
  <property>
    <name>hbase.wal.provider</name>
    <value>filesystem</value>
  </property>
  <property>
    <name>hbase.hregion.memstore.mslab.enabled</name>
    <value>true</value>
  </property>
</configuration>
```

+ `hbase.regionserver.global.memstore.size = 0.4` caps total MemStore memory at 40% of heap; the rest serves the block cache and reads.
+ Lower `hbase.hregion.memstore.flush.size` when write bursts are spiky; larger values reduce flush frequency but increase heap pressure.
+ `hbase.hregion.max.filesize` controls when a region splits; 10 GB is a common production value.
+ `hbase.hstore.blockingStoreFiles` makes the RegionServer block writes when too many store files pile up, which pushes back on the client instead of failing.

## How do you read and write data from applications?

The Java client is the canonical API. Create the `Connection` once per process and reuse it; `Table` and `Admin` are cheap to obtain from it. Batch and async paths are used for high throughput.

```java
Configuration conf = HBaseConfiguration.create();
conf.set("hbase.zookeeper.quorum", "zk1,zk2,zk3");
```

Rules that matter under load: never build a `Connection` per request, set `Scan.setCaching` (rows per RPC) and `setBatch` (cells per RPC), keep row keys short, and prefer `BufferedMutator` over many single `put` calls.

HappyBase talks to the Thrift gateway (`hbase-daemon.sh start thrift`). It is convenient for services and scripts, but the Thrift gateway is an extra hop compared with the native Java API.

```python
import happybase

connection = happybase.Connection("hbase-thrift.example.com", port=9090)
table = connection.table("app:impressions")

# Put a row
table.put(
    b"u42|9999999999999-1700000000000|a1",
    {b"i:campaign_id": b"cmp-99", b"i:amount": b"42.10"},
)

# Point read
row = table.row(b"u42|9999999999999-1700000000000|a1")
print(row.get(b"i:campaign_id"))

# Prefix scan for the most recent impressions of a user
for key, data in table.scan(row_prefix=b"u42|", limit=10):
    print(key, data)

# Atomic counter
counters = connection.table("app:counters")
counters.counter_inc(b"daily|2026-10-04", b"c:impressions", 1)

connection.close()
```

Use `happybase.ConnectionPool` in a web service so connections are reused, and keep the Thrift server close to the RegionServers to avoid an extra network round trip per scan.

## How do you query HBase with Apache Phoenix SQL?

Phoenix adds a SQL layer over HBase: it compiles SQL to HBase scans, maintains secondary indexes in HBase tables, and gives JDBC access. The row key is still the primary key, and the primary key order defines the physical sort order.

```sql
-- Primary key maps to the HBase row key; the leading columns drive the scan
CREATE TABLE IF NOT EXISTS impressions (
    user_id   VARCHAR NOT NULL,
    ts        TIMESTAMP NOT NULL,
    campaign  VARCHAR,
    amount    DECIMAL(10,2),
    CONSTRAINT pk PRIMARY KEY (user_id, ts DESC)
);

UPSERT INTO impressions (user_id, ts, campaign, amount)
VALUES ('u42', CURRENT_TIME(), 'cmp-99', 42.10);

-- Secondary index for queries that do not lead with the primary key
CREATE INDEX impressions_campaign_idx ON impressions (campaign);

SELECT campaign, COUNT(*), SUM(amount)
FROM impressions
WHERE user_id = 'u42'
GROUP BY campaign;
```

Phoenix is the right answer when interviewers ask "does HBase support SQL?" - **not natively, but Phoenix provides it**. Beware that a Phoenix query that does not lead with the primary key may fall back to a full scan unless a secondary index exists.

## How do HBase and an RDBMS differ?

Schema: RDBMS defines tables and typed columns up front. HBase defines tables and column families, but column qualifiers inside a family are dynamic and untyped (everything is bytes), so the schema is much looser.

Normalization: an RDBMS is normalized; HBase is deliberately denormalized and stores wide, sparse rows.

Partitioning: an RDBMS requires manual partitioning and sharding. HBase splits a table into regions automatically as it grows, and can be pre-split manually.

Other differences: RDBMS supports joins, multi-row transactions, and SQL; HBase supports single-row atomicity, no joins, and no server-side SQL without Phoenix. HBase tables are sparse - a row only stores the cells that exist.

### How do Hive and HBase differ?

Apache Hive is a data warehousing layer on top of Hadoop. It queries data in HDFS with HiveQL (a SQL-like language) that is translated into MapReduce, Tez, or Spark jobs, and it performs batch processing.

Apache HBase is a NoSQL wide-column key/value store on top of HDFS. Unlike Hive, HBase operations run in real time against its own database rather than as batch jobs. HBase splits tables into regions and groups columns into column families.

Hive and HBase are complementary: Hive is good for analytical scans and joins, HBase for real-time key-based access. Data can be read and written between them, for example by storing HBase tables in Hive or writing Hive results into HBase.

## How does the write path work?

The client does not write directly to HFiles. It first appends the mutation to the WAL, then applies it to the MemStore of the target region. The MemStore is flushed to disk as an HFile when it reaches a threshold or the region is closed, and the WAL entries for those edits can then be discarded. Region location comes from the client's cache of `hbase:meta`.

```text
put -> find region (meta cache) -> RegionServer -> WAL append -> MemStore -> (flush) HFile
```

**Mechanics.** A client write is a pipeline: resolve the region from the cached `hbase:meta` map, send the mutation to the owning RegionServer, append to the WAL, apply to the region's MemStore, acknowledge. The client never writes an HFile. Flushing happens later, either when a MemStore crosses `hbase.hregion.memstore.flush.size`, when the server's global MemStore limit is reached, or when the region is closed or moved.

```text
put -> meta cache lookup -> RegionServer -> WAL append -> MemStore -> ack
                                            (later) flush -> HFile on HDFS
```

**Worked example.** With a 128 MB flush threshold, a region takes roughly 128 MB of writes between flushes. At 5,000 writes per second of 200 bytes each, that is about 1 MB/second per region, so a flush every two minutes and a new HFile per flush.

**Common mistake.** Assuming the write is durable once it is in the MemStore. It is durable once the WAL append is synced; MemStore-only data is lost on crash unless the WAL replays it. The other common error is expecting a put to be fast without bound: when the MemStore is over the global limit, the RegionServer blocks writes until a flush completes.

### What are the WAL and MultiWAL?

The WAL (Write-Ahead Log), historically called HLog, records every mutation before it is applied to memory. It is a Hadoop sequence file of `WALKey` plus cells, and it is used to replay not-yet-flushed data after a server crash. On RegionServer failure, the master splits the WAL by region and replays each region's edits so committed writes are not lost.

```properties
# Durability settings in hbase-site.xml
hbase.wal.provider = filesystem
# ASYNC_WAL is faster but can lose recent writes on crash; SYNC_WAL is the default
```

**Mechanics.** The WAL is the recovery contract: a write is acknowledged only after the WAL append. If the RegionServer dies before the MemStore is flushed, the master learns of the failure from ZooKeeper, splits the dead server's WALs by region, and assigns the regions to new servers, which replay their edits before serving. Sequence numbers let replay skip anything already flushed.

**Worked example.** A server with 4 GB of unflushed MemStore crashes at 12:00:00. Recovery splits the WAL, replays each region's edits on the new owner, and by 12:00:30 the regions are serving with the acknowledged writes present. Clients see a short retry window, not data loss.

**Common mistake.** Assuming recovery is instantaneous or invisible. The WAL for a busy server can be gigabytes, and log splitting plus replay takes time proportional to that size, which is why region count per server, flush tuning, and MultiWAL grouping matter for failover time. Writes made with `ASYNC_WAL` or `SKIP_WAL` are genuinely lost.

With a single WAL per RegionServer, the RegionServer must write to the WAL serially because HDFS files must be sequential. That makes the WAL a throughput bottleneck. MultiWAL gives each RegionServer multiple WAL files so different regions can write concurrently, improving write throughput.

```xml
<property>
  <name>hbase.wal.regiongrouping.strategy</name>
  <value>identity</value>
</property>
<property>
  <name>hbase.wal.provider</name>
  <value>filesystem</value>
</property>
```

### What is log splitting?

When a RegionServer dies, the edits belonging to its regions must be recovered from the WAL. The edits in the WAL are grouped by region so each set can be replayed to rebuild that region's data. The process of grouping and distributing WAL edits by region is called log splitting.

## How does HBase delete data?

When you delete a cell, the data is not immediately removed; a tombstone marker is written that makes the deleted cells invisible, and the data is physically removed during compaction.

+ Version delete marker: marks a single version of a column.
+ Column delete marker: marks all versions of one column.
+ Family delete marker: marks all columns in a column family.

**Mechanics.** A delete is a write. HBase inserts a tombstone cell at the delete's timestamp, and reads discard any cell at or below it. Tombstones come in three scopes: a version marker for one version, a column marker for all versions of one qualifier, and a family marker for every qualifier in a family (used by `deleteall`). Physical removal happens only when a compaction rewrites the affected files.

**Worked example.** `delete 'app:impressions', 'u42|1', 'i:amount'` writes a column tombstone. A later `get` returns no `i:amount` even though the value bytes are still in the HFile, and the bytes disappear at the next major compaction. `count` can overstate the visible row count until tombstones are compacted, because it counts keys, not visible cells.

**Common mistake.** Deleting with an explicit old timestamp and then writing a newer value with an older timestamp. The tombstone masks the new value until a major compaction clears it. Let the server assign timestamps unless you have a specific reason not to.

If you delete data and then insert new data with an earlier timestamp than the tombstone, subsequent Gets may still be masked by the tombstone and will not see the new value until a major compaction runs. This is why clients should write timestamps that move forward.

## How do filters work?

Filters perform server-side filtering for scans, so rows and cells are discarded on the RegionServer before crossing the network. They were introduced in HBase 0.92 and can be used from the shell, the Java API, and Thrift. Because filtering happens after the scan reads the row from the store, filters reduce network and client work but do not eliminate the underlying I/O - a well-designed row key is still better than a filter.

```bash
# Shell examples: prefix, single-column value, and a filter list
scan 'app:impressions', {FILTER => "PrefixFilter('u42|')", LIMIT => 10}
scan 'app:impressions', {FILTER => "SingleColumnValueFilter('i', 'campaign_id', =, 'binary:cmp-99')"}
scan 'app:impressions', {FILTER => "RowFilter(>=, 'binary:u42|') AND ColumnPrefixFilter('camp')"}
scan 'app:impressions', {FILTER => "PageFilter(100)"}
```

In [ ]:
def single_column_value(column: str, value: str):
    """Like SingleColumnValueFilter: evaluated on the server, inside the scan loop."""
    return lambda row, cells: cells.get(column, [(None, None)])[0][1] == value


hits = t.scan(start="u4", stop="u5", now=3_000, row_filter=single_column_value("i:campaign_id", "cmp-99"), limit=50)
assert [row for row, _ in hits] == ["u43|a7"]               # u42's cmp-99 version was deleted
print(hits)

The same scan with HappyBase passes the filter string through to the RegionServer:

```python
# HappyBase: a filter string is passed through to the RegionServer
for key, data in table.scan(
    row_prefix=b"u42|",
    filter=b"SingleColumnValueFilter('i', 'campaign_id', =, 'binary:cmp-99')",
    limit=50,
):
    print(key, data)
```

**Mechanics.** A filter is evaluated on the RegionServer inside the scan loop. `Filter.filterRowKey` runs before the row is read and can skip it cheaply; `filterCell` runs after cells are loaded and can drop them but not the I/O that read them. That split is why a `RowFilter` or `PrefixFilter` is much cheaper than a `SingleColumnValueFilter`: the first can skip a row before reading its cells, the second must read the value to decide.

**Worked example.** `PrefixFilter('u42|')` skips whole regions and rows using the key range alone. `SingleColumnValueFilter('i','campaign_id',=,'binary:cmp-99')` has to load each row's `campaign_id` cell, so it costs roughly a full scan of the key range before the filter is applied.

**Common mistake.** Using a value filter as a substitute for row key design. Filters reduce network transfer and client work, not the underlying store I/O. If a value predicate runs on every query, encode that value into the row key or build a secondary index table.

## How does compaction work?

Under heavy writes it is impractical to keep one file per store, because every read would seek across many files. Compaction merges HFiles to reduce the number of disk seeks per read and to drop deleted or expired cells. Choosing when and how aggressively to compact is a central HBase tuning topic.

There are two types. In minor compaction, a small set of adjacent HFiles is merged into one without removing the deleted data, and the files to merge are chosen by policy. In major compaction, all HFiles of a region's column family are merged into one, deleted and expired cells are discarded, and it is often triggered manually or by a schedule.

```bash
# Trigger a major compaction on a region (shell) or a single table
major_compact 'app:impressions'
```

Major compaction rewrites a lot of data, so it should be scheduled off-peak and rate-limited with `hbase.hstore.compaction.throughput.lower.bound` and related settings.

**Mechanics.** Minor compaction is incremental: it picks adjacent files by the compaction policy, merges them, and leaves tombstones because it cannot prove no other file holds data the tombstone masks. Major compaction is total: all store files of a region's family merge into one, and deletes, TTL-expired cells, and surplus versions are dropped. Major compaction is also the point at which a change like block size, compression, or family TTL is fully materialized.

```bash
major_compact 'app:impressions'              # whole table
major_compact 'app:impressions', 'encoded_region_name'
compact 'app:impressions'                    # enqueue a minor compaction
```

**Worked example.** A region with 30 store files and heavy deletes is read-slow. A minor compaction leaves tombstones and does not reduce the file count much. A scheduled major compaction merges to one file, drops the tombstones, and the read starts hitting a single index and Bloom filter.

**Common mistake.** Major-compacting everything at once on a busy cluster. Major compaction rewrites the whole region, so a cluster-wide sweep saturates disks and can cause write stalls. Use the major compaction interval with jitter, or trigger per-region during low traffic, and rate-limit with the throughput bounds.

Compaction and splitting are the two background processes that keep a growing table healthy. Compaction controls read amplification and storage; splitting controls distribution across RegionServers. Both are usually automatic, but the commands and policies matter in interviews.

+ Schedule major compactions off-peak and jitter them, because they rewrite the whole region.
+ On a write-heavy table, tune `hbase.hstore.compaction.min` and the throughput bounds rather than disabling compaction.
+ Pre-split new tables and use `SteppingSplitPolicy` to avoid split storms, where one split immediately triggers the next.
+ A region that stays hot after its split is a row key design problem, not a compaction problem.

## How do you implement joins in HBase?

HBase does not support joins directly. Use a MapReduce or Spark job that scans the relevant tables and joins them, or denormalize at write time so that the data needed together is stored in one row (the preferred approach for a serving path).

```python
# Spark can read HBase through the HBase-Spark connector and join to curated tables
impressions = (
    spark.read.format("org.apache.hadoop.hbase.spark")
    .option("hbase.table", "app:impressions")
    .option("hbase.columns.mapping", "rowkey STRING :key, campaign STRING i:campaign_id, ts STRING i:ts")
    .load()
)
campaigns = spark.table("curated.campaigns")
joined = impressions.join(campaigns, impressions.campaign == campaigns.campaign_id)
```

The interview point: a join in a serving path means the schema is wrong. Reshape the row key and duplicate the small side into the row instead.

## What is a Bloom filter in HBase?

A Bloom filter is a space-efficient structure used to test whether an HFile contains a certain row or row-column cell, which avoids reading HFiles that cannot contain the requested key. HBase supports `ROW` (test row key only) and `ROWCOL` (test row key plus column) Bloom filters per column family, which improves read throughput for random point reads.

```bash
# Add a Bloom filter to a family, or tune it at create time
alter 'app:impressions', {NAME => 'i', BLOOMFILTER => 'ROWCOL'}
```

Bloom filters help point reads and hurt nothing on writes except a little memory and file size; they do not help scans of many rows.

## What else should you know about HFiles and block size?

When you alter the block size of a column family, newly written data uses the new block size while old data stays in blocks of the old size. Existing data is still read correctly, and new files flushed after the change use the new block size. All data is transformed to the new block size only after the next major compaction.

You can use the `HFile.main()` method (the `hbase org.apache.hadoop.hbase.io.hfile.HFile` tool) to inspect an HFile without a running HBase cluster.

```bash
hbase org.apache.hadoop.hbase.io.hfile.HFile -p -f /hbase/data/default/app_impressions/region/hfile
```

`HBaseFsck` implements the `hbck` tool, which offers command-line switches to check region consistency and table integrity and to repair a corrupted HBase. It runs in a read-only inconsistency-identifying mode and a multi-phase read-write repair mode.

```bash
hbase hbck
hbase hbck -details
hbase hbck -repair
```

## What are the block caches?

HBase provides two BlockCache implementations: the on-heap `LruBlockCache` (the default) and `BucketCache`, which is usually off-heap. A common production configuration is a combined cache: an LRU cache in front of a `BucketCache` backed by off-heap memory, SSD, or a file, to keep the JVM heap small and avoid GC pauses.

```xml
<property>
  <name>hbase.bucketcache.ioengine</name>
  <value>offheap</value>
</property>
<property>
  <name>hbase.bucketcache.size</name>
  <value>16384</value>
</property>
<property>
  <name>hfile.block.cache.size</name>
  <value>0.2</value>
</property>
```

## What is TTL in HBase?

TTL (time to live) is a data retention setting at the column family level. It specifies how long a cell version is kept; once a version's timestamp is older than the TTL, it is removed during compaction and is not returned by reads.

```bash
create 'app:impressions', {NAME => 'i', TTL => 7776000}   # 90 days in seconds
```

## How do you pre-split and split regions?

Tables are initially created with one region by default. During bulk import, all clients write to that single region until it grows large enough to split, which serializes the load and slows the import. Pre-creating empty regions (pre-splitting) spreads the initial write load across the cluster from the start.

```bash
create 'app:impressions', {NAME => 'i'}, SPLITS => ['10', '20', '30', '40']
```

Manual region splitting is done when there is an unexpected hotspot caused by many clients querying the same region, or when you know the key distribution ahead of time and want to pre-split for bulk loading. Automatic splitting handles normal growth.

```bash
# Split a table at an explicit key, and merge two adjacent regions
split 'app:impressions', '20'
merge_region 'ENCODED_REGION_A', 'ENCODED_REGION_B'
```

**Mental model.** A region is the unit of distribution, so region count per server sets the parallelism and the number of MemStores, and region size sets how long a split and a compaction take. Too few regions means one server cannot use its disks and CPUs; too many small regions means many small MemStores, constant flushing, and a heavy compaction queue.

**How it works.** `hbase.hregion.max.filesize` is the store size at which a region splits, so target region size is roughly that value. Region count follows from the data size: `regions = ceil(stored bytes / target region size)`. Pre-splitting passes explicit boundaries at create time, and the boundaries must match the leading bytes of the real row keys. The split policy decides how a daughter splits later; `SteppingSplitPolicy` halves the threshold at each generation, which spreads a fast-growing table quickly, while `IncreasingToUpperBoundRegionSplitPolicy` grows the threshold with the region count.

```xml
<property><name>hbase.hregion.max.filesize</name><value>10737418240</value></property>
<property><name>hbase.regionserver.region.split.policy</name>
  <value>org.apache.hadoop.hbase.regionserver.SteppingSplitPolicy</value></property>
<property><name>hbase.regionserver.regionSplitLimit</name><value>1000</value></property>
```

**Worked example.** 2 billion rows at 400 bytes before compression and a 0.4 compression ratio is 320 GB stored. At a 10 GiB target, that is 30 regions, so pre-split with 29 boundaries spread evenly over the salt range, for example `['10','20',...,'e0']` for a two-character hex salt.

**Common mistake.** Pre-splitting with even byte boundaries when the key is not uniformly distributed across those bytes. If 90 percent of keys start `0`, even boundaries produce one huge region and several empty ones. Derive boundaries from the actual key histogram or from the salt range, and remember that splits and pre-splits both trigger compaction of the new daughters.

```bash
# Pre-split on the salt range, then verify the region chain
create 'iot:readings', {NAME => 'd', VERSIONS => 1, TTL => 7776000}, \
  SPLITS => ['04','08','0c','10','14','18','1c','20','24','28','2c','30']
list_regions 'iot:readings'

# Force a split of one region at an explicit key when a range outgrows its neighbours
split 'iot:readings', '18'
# Merge two small adjacent regions back after a load spike
merge_region 'ENCODED_REGION_A', 'ENCODED_REGION_B'
```

## How does HBase support bulk loading?

Bulk loading writes HFiles directly, bypassing the WAL and MemStore, which makes it much faster than `put` for large imports. Two steps:

1. Generate HFiles from the source data with a MapReduce or Spark job (`ImportTsv` or a custom job) in HBase's internal format.
2. Load the prepared files into a running cluster with `completebulkload` / `LoadIncrementalHFiles`, which moves each file into the correct region.

```bash
# 1) Convert a TSV on HDFS into HFiles
hbase org.apache.hadoop.hbase.MapReduce.ImportTsv \
  -Dimporttsv.columns=HBASE_ROW_KEY,i:campaign_id,i:amount \
  -Dimporttsv.bulk.output=/tmp/impressions_hfiles \
  app:impressions /data/incoming/impressions.tsv

# 2) Load the HFiles into the table (pre-split it first for balance)
hbase org.apache.hadoop.hbase.MapReduce.LoadIncrementalHFiles \
  /tmp/impressions_hfiles app:impressions
```

Pre-split the table before a bulk load, because `LoadIncrementalHFiles` assigns each HFile to the region that owns its key range.

## How does HBase provide high availability?

+ Region replication: each region can have replicas served by different RegionServers, and the load balancer avoids co-hosting a replica set.
+ WAL replay: committed edits are recovered from the WAL after a RegionServer crash.
+ Multiple masters: backup H Masters are elected so the master is not a hard single point of failure.
+ HDFS replication: the underlying WALs and HFiles are stored with HDFS replication.
+ For cross-cluster DR, HBase replication ships edits from a source cluster to a peer cluster.

## How do you diagnose a hotspot?

**Mental model.** HBase has exactly one index: the row key. A region owns a contiguous key range, and the RegionServer hosting that region owns all traffic for that range. Even distribution is therefore a property of the key bytes, not of the table or the cluster. Anything that increases monotonically concentrates load in the last region; anything with a head byte that is spread over the key space distributes load.

**How it works.** Three standard transforms. Salting prepends a fixed-width bucket derived from the entity id, so writes for an entity always land in the same known bucket and different entities spread across buckets. Hashing replaces a natural key with a digest, which spreads writes but destroys range scans. Reverse timestamps make the newest row sort first inside a known prefix, which turns "latest N" into a bounded scan. Pick the transform from the query pattern.

```python
import hashlib

def salted_key(entity_id: str, ts_ms: int, buckets: int = 32) -> str:
    salt = int(hashlib.md5(entity_id.encode()).hexdigest(), 16) % buckets
    return f"{salt:02d}|{entity_id}|{9_999_999_999_999 - ts_ms:013d}"

def hotspot_ratio(counts: dict) -> tuple:
    total = sum(counts.values())
    average = total / len(counts)
    busiest = max(counts, key=counts.get)
    return busiest, counts[busiest] / average
```

**Worked example.** A keyset of sequential device ids produces a perfectly even write distribution once salted, but a read for one device must compute the entity's bucket (`md5(id) % 32`) rather than scanning all 32. If the caller cannot compute the salt, store an index row that maps the entity to its bucket, or accept a 32-way fan-out scan.

**Common mistake.** Salting with too few buckets, or with a bucket derived from something that does not exist at read time. Two or four buckets just move the hotspot; a salt the reader cannot reproduce turns every lookup into a full-table scan.

**Diagnosing a hotspot.** Compare per-region request counts. A region above roughly three times the cluster average, or one server carrying far more regions than its peers, is the suspect. Look at `writeRequestCount` and `readRequestCount` per region, not just cluster totals.

```bash
# Find the region ranges and their server, then compare request rates
list_regions 'app:impressions'
status 'detailed'

# Ask one RegionServer for its region-level metrics over JMX
curl -s "http://rs1.example.com:16030/jmx?qry=Hadoop:service=HBase,name=RegionServer,sub=Regions" | head

# If a single sequential region is hot, split it; if one key is hot, splitting will not help
split 'app:impressions', '40'
```

## How do you tune the read path?

**Mental model.** A read merges the region's MemStore with every store file it might need, applying Bloom filters and the block cache to avoid touching files that cannot hold the key. Read latency therefore scales with the number of store files, the block cache hit ratio, and how precisely the request is expressed. Tuning the read path means giving the cache the right working set, giving the reader the right key range, and keeping the file count low.

**How it works.** The block cache holds HFile blocks, so its size relative to the hot working set decides most point-read latency. Bloom filters skip files per key. `Scan.setCaching` sets rows per RPC and `Scan.setBatch` sets cells per RPC, trading memory and round trips. A bounded `STARTROW`/`ENDROW` plus a filter that examines the key is far cheaper than a value filter, because value filters must read the value first.

```xml
<property><name>hfile.block.cache.size</name><value>0.2</value></property>
<property><name>hbase.bucketcache.ioengine</name><value>offheap</value></property>
<property><name>hbase.bucketcache.size</name><value>16384</value></property>
```

```java
Scan scan = new Scan()
    .withStartRow(Bytes.toBytes("07|u42|"))
    .withStopRow(Bytes.toBytes("07|u42|￿".getBytes()))
    .setCaching(200)     // rows per RPC
    .setBatch(100)       // cells per RPC
    .setLimit(1000);     // stop across regions
scan.addFamily(Bytes.toBytes("i"));   // never read families you do not need
```

**Worked example.** A 50-row page over a key range costs one RPC per region when `setCaching(200)`, versus one RPC per row with the default caching of 1. On a 60-region table, the difference is 60 round trips instead of 3,000 for the same page.

**Common mistake.** Raising caching without bound. Large caching means the server holds many rows in memory per scanner, and too many concurrent shallow scanners exhaust the RegionServer. Match caching to row size and limit the number of open scanners. A second mistake is using `BLOCKSIZE` or Bloom filters to fix a scan-latency problem that is really too many store files; compaction and a bounded range fix that.

```sql
-- Phoenix: confirm the query becomes a range scan, not a full scan
EXPLAIN SELECT campaign, amount FROM impressions WHERE user_id = 'u42' AND ts > 1700000000000;

-- A query that does not lead with the primary key needs a secondary index
CREATE INDEX IF NOT EXISTS impressions_campaign_idx ON impressions (campaign) INCLUDE (amount);

-- Cover the query so the index table alone answers it
SELECT campaign, SUM(amount) FROM impressions WHERE campaign = 'cmp-99' GROUP BY campaign;
```

## How do you tune the write path?

**Mental model.** A write is a WAL append plus a MemStore insert, acknowledged before anything reaches an HFile. Throughput is limited by WAL serialization, by MemStore budgets and flush behavior, and by files piling up behind compaction. When the global MemStore limit is crossed, the RegionServer blocks writes, so a compaction backlog eventually becomes a latency problem, not a disk problem.

**How it works.** `hbase.regionserver.global.memstore.size` caps total MemStore use and, when the lower limit is crossed, forces flushes. `hbase.hregion.memstore.flush.size` sets the per-region flush threshold. `hbase.hstore.blockingStoreFiles` blocks writes for a region when too many files accumulate. `hbase.hregion.majorcompaction` schedules major compaction and `hbase.hregion.majorcompaction.jitter` spreads it. WAL compression and MultiWAL grouping reduce the append bottleneck. For a custom loader, `HFileOutputFormat2.configureIncrementalLoad` partitions the reducer output by region boundaries, so each reducer writes HFiles that belong to one region and can be moved in by `LoadIncrementalHFiles`.

```xml
<property><name>hbase.regionserver.global.memstore.size</name><value>0.4</value></property>
<property><name>hbase.regionserver.global.memstore.size.lower.limit</name><value>0.95</value></property>
<property><name>hbase.hregion.memstore.flush.size</name><value>134217728</value></property>
<property><name>hbase.hstore.blockingStoreFiles</name><value>16</value></property>
<property><name>hbase.hregion.majorcompaction</name><value>604800000</value></property>
<property><name>hbase.hregion.majorcompaction.jitter</name><value>0.5</value></property>
<property><name>hbase.wal.regiongrouping.numgroups</name><value>4</value></property>
<property><name>hbase.regionserver.wal.enablecompression</name><value>true</value></property>
```

**Worked example.** A 32 GB heap with a 0.4 MemStore fraction allows about 12.8 GB of MemStore. At a 128 MB flush threshold that is roughly 100 region MemStores of headroom. If the table has 400 regions, average MemStore size must stay near 32 MB or flushes become continuous, which is why region count per server is a memory decision.

**Common mistake.** Raising `hbase.hregion.memstore.flush.size` to reduce flush frequency without checking the global fraction. Larger per-region MemStores mean fewer regions fit in the budget, and the server starts blocking writes sooner under a burst. The other mistake is disabling major compaction to protect throughput and never scheduling it, so tombstones and expired cells accumulate and read amplification grows without bound.

```bash
# Load a large export with HFiles, bypassing the WAL and MemStore
hbase org.apache.hadoop.hbase.mapreduce.ImportTsv \
  -Dimporttsv.separator=, \
  -Dimporttsv.columns=HBASE_ROW_KEY,i:campaign_id,i:amount \
  -Dimporttsv.bulk.output=/tmp/impressions_hfiles \
  app:impressions /data/incoming/impressions.tsv

# Pre-split first, then move the HFiles into the owning regions
hbase org.apache.hadoop.hbase.mapreduce.LoadIncrementalHFiles /tmp/impressions_hfiles app:impressions

# Verify balance and trigger maintenance
list_regions 'app:impressions'
compact 'app:impressions'
major_compact 'app:impressions'
```

## Things to remember

| Topic | Fact | Why it matters |
|---|---|---|
| Row key | byte-sorted; regions split on key ranges | a sequential key (timestamp-first) writes to one region — hotspotting |
| Write path | WAL first, then memstore; memstore flush writes HFiles | the WAL is what survives a RegionServer crash |
| Read path | block cache -> memstore -> HFiles with Bloom filters skipping misses | Bloom filters and block cache are why point reads stay fast |
| Deletes | tombstone markers; space reclaimed at major compaction | a delete is a write until compaction removes it |
| Compaction | minor merges HFiles; major rewrites a store fully | major compactions are IO storms; schedule them off-peak |
| Regions | pre-split empty tables; one region serves one key range per server | pre-splitting before a bulk load avoids the single-region collapse |
| TTL/versions | per column family; versions bounded by keeping count or TTL | unbounded versions turn time series into unbounded rows |
| SQL | none natively; Apache Phoenix adds SQL on top | HBase itself is get/scan/put/delete |

## Real world use cases for HBase

### 1. IoT device telemetry time series

A utility monitors 60,000 smart meters emitting a reading every 15 seconds. The serving API needs the latest readings for a device and a time-range view over the last day, and the raw history must be retained for 90 days.

Business context: outage detection and customer-facing usage dashboards read a single device at a time. Data volume: about 4,000 writes/second, roughly 200 GB/month after compression.

Row key design: `salt|device_id|reverse_timestamp`, where salt is a fixed hash bucket per device and the reverse timestamp makes the newest reading sort first. TTL on the column family drops data after 90 days.

```bash
# Table with a salted key, one version, 90-day TTL, and a row+column Bloom filter
create 'iot:readings', {NAME => 'd', VERSIONS => 1, TTL => 7776000, BLOOMFILTER => 'ROWCOL', COMPRESSION => 'SNAPPY'}, SPLITS => ['04', '08', '0c', '10', '14', '18', '1c', '20']
```

Why this design: the salt spreads writes across all 32 buckets so no single region takes the load, the device prefix within a bucket makes the latest-reading Get a single point read, and the reverse timestamp makes `scan row_prefix=salt|device` return newest first with a `LIMIT`.

### 2. Clickstream profile store for real-time bidding

An ad-tech platform records 2 billion impressions per day and must answer "what did this user see in the last 30 minutes" in under 10 ms during an auction.

Business context: the bidder enriches a request with a user's recent behavior. Data volume: about 25,000 writes/second, 2 billion rows/day, 30-minute working set.

Row key design: `salt|user_id|reverse_timestamp` for the recent window, plus a separate compacted table `user_profile` keyed by `salt|user_id` for the latest aggregates. The recent table uses a short TTL and the serving table keeps one version per key.

```java
Table recent = conn.getTable(TableName.valueOf("rtb:user_events"));
Scan scan = new Scan()
    .withStartRow(Bytes.toBytes(saltOf(userId) + "|" + userId + "|"))
    .withStopRow(Bytes.toBytes(saltOf(userId) + "|" + userId + "|\uFFFF".getBytes()))
    .setLimit(200)
    .setCaching(200);
```

Why this design: point reads and short scans are exactly HBase's strength, the reverse timestamp gives the newest events first without sorting, and the narrow row key keeps the block cache effective. Analytics still runs on a Parquet copy, not on HBase.

### 3. Messaging and conversation history

A messaging platform stores chat history so a client can open a conversation and load the most recent messages, then page backwards.

Business context: 40 million monthly users, peak 50,000 messages/second, per-conversation ordering required. Data volume: several TB per month.

Row key design: `conversation_id|reverse_timestamp|message_id`. Because a conversation is a natural read unit, the design is tall-and-thin with one row per message; the reverse timestamp gives pagination in both directions (newest first, and older pages by advancing the start row).

```bash
# Newest 50 messages in a conversation
scan 'msg:history', {STARTROW => 'conv-88217|', ENDROW => 'conv-88217|\xff', LIMIT => 50}
```

Why this design: ordering by row key replaces an ORDER BY, pagination is a bounded scan, and atomic increments on a companion counters table provide unread counts without a transaction.

### 4. Wide sparse product catalog with per-attribute updates

A retailer stores product attributes where different categories have wildly different fields (electronics have voltages, apparel has sizes), and the pricing service updates individual attributes frequently.

Business context: the storefront reads one product at a time and updates a few attributes per event. Data volume: 50 million products, 10-40 attributes each, low-latency reads.

Row key design: `salt|sku`, with attributes as dynamic qualifiers in a single family. Sparse storage means a product only occupies the qualifiers it actually has.

```bash
put 'catalog:products', '07|sku-99321', 'p:color', 'black'
put 'catalog:products', '07|sku-99321', 'p:voltage', '220'
get 'catalog:products', '07|sku-99321'
```

Why this design: a relational schema would need dozens of sparse columns or an entity-attribute-value table with joins; HBase stores sparse rows natively and reads a whole product in one Get. Salt keeps the catalog write load balanced during bulk price updates.

### 5. Streaming feature store for fraud detection

A payments company computes features (transaction velocity, recent declines) in a stream processor and serves them to a scoring service with single-digit-millisecond latency.

Business context: the scorer runs on every authorization and cannot wait for a warehouse. Data volume: 8,000 events/second, features for 30 million cards.

Row key design: `salt|card_id` for the current feature vector, with the feature values in one family and short TTLs on the fast-moving counters. A separate `salt|card_id|reverse_timestamp` table holds the recent event trail.

```python
# Stream processor writes the latest feature vector; scorer reads it
table.put(
    (salt(card_id) + "|" + card_id).encode(),
    {
        b"f:txn_count_1h": str(count_1h).encode(),
        b"f:declines_24h": str(declines_24h).encode(),
        b"f:updated_at": str(now_ms).encode(),
    },
)
```

Why this design: HBase gives a durable, horizontally scalable, low-latency serving layer for a keyed feature vector, writes are single-row atomic so the vector is never half-updated, and the analytics side of the feature computation lives in Kafka plus Flink rather than in HBase.

## HBase exercises

### Try it 1. Design a salted row key from key patterns

**Task.** Build a salted, reverse-timestamp row key for a set of device ids, verify that the salt is fixed width, that the key is deterministic, and that the ids spread over several buckets.

**Hint.** Derive the salt with `hashlib.md5(entity_id.encode())` modulo the bucket count, format it as `%02d`, and reverse the timestamp with `9_999_999_999_999 - ts_ms`.

**Expected result.** Every key starts with a two-digit salt below 16, the keys are unique, and at least 6 of the 16 buckets are used.

In [ ]:
import hashlib

BUCKETS = 16


def salted_key(entity_id, ts_ms):
    salt = int(hashlib.md5(entity_id.encode()).hexdigest(), 16) % BUCKETS
    return f"{salt:02d}|{entity_id}|{9_999_999_999_999 - ts_ms:013d}"


device_ids = [f"device-{i:04d}" for i in range(1, 17)]
row_keys = [salted_key(device_id, 1_700_000_000_000 + i) for i, device_id in enumerate(device_ids)]

for device_id, row_key in zip(device_ids, row_keys):
    print(f"{device_id} -> {row_key}")

salts = [row_key[:2] for row_key in row_keys]
print("distinct buckets used:", len(set(salts)))

assert all(len(salt) == 2 and 0 <= int(salt) < BUCKETS for salt in salts)
assert len(set(row_keys)) == len(device_ids)
assert salted_key("device-0001", 1_700_000_000_000) == row_keys[0]
assert len(set(salts)) >= 6

### Try it 2. Compute region count from target region size

**Task.** Given a row count, an average row size, a compression ratio, and a target region size, compute the stored bytes, the region count, and the number of pre-split boundaries.

**Hint.** Stored bytes are `rows * avg_row_bytes * compression_ratio`. Use ceiling division `-(-a // b)` so a partial region still counts, and pre-split boundaries are `regions - 1`.

**Expected result.** `320,000,000,000` stored bytes (about `298 GiB`), 30 regions, and 29 pre-split boundaries.

In [ ]:
ROWS = 2_000_000_000
AVG_ROW_BYTES = 400
COMPRESSION_RATIO = 0.4
TARGET_REGION_BYTES = 10 * 1024 ** 3

stored_bytes = int(ROWS * AVG_ROW_BYTES * COMPRESSION_RATIO)
regions = int(-(-stored_bytes // TARGET_REGION_BYTES))

print(f"stored bytes   = {stored_bytes:,}")
print(f"stored GiB     = {stored_bytes / 1024 ** 3:.1f}")
print(f"regions        = {regions}")
print(f"pre-split keys = {regions - 1}")

assert stored_bytes == 320_000_000_000
assert regions == 30
assert regions - 1 == 29

### Try it 3. Detect a hot region from request counts

**Task.** Given per-region request counts, compute the cluster average, the busiest region's ratio to the average, and the list of regions that exceed twice the average.

**Hint.** Ratio is `count / average`. Use `max(counts, key=counts.get)` for the busiest region and a list comprehension for the outliers.

**Expected result.** The busiest region is `region-a` at about `2.97x` the average, and it is the only region above the `2.0x` threshold.

In [ ]:
request_counts = {
    "region-a": 50_000,
    "region-b": 20_000,
    "region-c": 18_000,
    "region-d": 12_000,
    "region-e": 10_000,
    "region-f": 9_000,
    "region-g": 8_000,
    "region-h": 7_500,
}

total = sum(request_counts.values())
average = total / len(request_counts)
ratios = {region: count / average for region, count in request_counts.items()}
busiest = max(ratios, key=ratios.get)
hot_regions = [region for region, ratio in ratios.items() if ratio > 2.0]

print(f"total requests = {total:,}")
print(f"average        = {average:,.1f}")
print(f"busiest        = {busiest} at {ratios[busiest]:.2f}x")
print("regions above 2.0x:", hot_regions)

assert total == 134_500
assert round(ratios[busiest], 2) == 2.97
assert hot_regions == ["region-a"]

### Try it 4. Size MemStore from a heap fraction

**Task.** A RegionServer has a 32 GB heap, a 0.4 global MemStore fraction, a 0.2 block cache fraction, and a 128 MB per-region flush threshold. Compute the MemStore budget, the block cache budget, the GC headroom, and how many region MemStores fit in the budget.

**Hint.** The fractions are relative to the heap, so the budgets are `heap * fraction`. Region MemStore capacity is `memstore_gb * 1024 // flush_size_mb`, and headroom is what is left after both budgets.

**Expected result.** 12.8 GB of MemStore, 6.4 GB of block cache, 12.8 GB of headroom, and 102 region MemStores at the flush threshold.

In [ ]:
HEAP_GB = 32
MEMSTORE_FRACTION = 0.4
BLOCK_CACHE_FRACTION = 0.2
FLUSH_SIZE_MB = 128

memstore_gb = HEAP_GB * MEMSTORE_FRACTION
block_cache_gb = HEAP_GB * BLOCK_CACHE_FRACTION
headroom_gb = round(HEAP_GB - memstore_gb - block_cache_gb, 1)
region_memstores = int(memstore_gb * 1024 // FLUSH_SIZE_MB)

print(f"memstore budget    = {memstore_gb:.1f} GB")
print(f"block cache budget = {block_cache_gb:.1f} GB")
print(f"GC headroom        = {headroom_gb:.1f} GB")
print(f"region MemStores   = {region_memstores} at {FLUSH_SIZE_MB} MB each")
print(f"flush size in bytes = {FLUSH_SIZE_MB * 1024 * 1024}")

assert memstore_gb == 12.8
assert block_cache_gb == 6.4
assert headroom_gb == 12.8
assert region_memstores == 102
assert FLUSH_SIZE_MB * 1024 * 1024 == 134_217_728

### Try it 5. Estimate scan cost from row count and caching

**Task.** Compare the RPCs and bytes of a full-table scan with a bounded prefix scan. The table has 50 million rows of 200 bytes each across 60 regions; the scan uses `setCaching(200)` and the bounded read stops after 1,000 rows.

**Hint.** RPCs are `ceil(rows / caching)` and bytes are `rows * row_bytes`. The bounded case uses the limit instead of the full row count.

**Expected result.** The full scan needs 250,000 RPCs and 10,000,000,000 bytes, the bounded scan needs 5 RPCs and 200,000 bytes, a reduction of 50,000 times.

In [ ]:
import math

ROWS = 50_000_000
ROW_BYTES = 200
CACHING = 200
REGIONS = 60
LIMIT = 1_000

full_bytes = ROWS * ROW_BYTES
full_rpcs = math.ceil(ROWS / CACHING)
bounded_bytes = LIMIT * ROW_BYTES
bounded_rpcs = math.ceil(LIMIT / CACHING)
reduction = full_rpcs // bounded_rpcs

print(f"rows per region      = {ROWS / REGIONS:,.0f}")
print(f"full scan            = {full_rpcs:,} RPCs, {full_bytes / 1024 ** 3:.1f} GiB")
print(f"bounded prefix scan  = {bounded_rpcs} RPCs, {bounded_bytes / 1024:.0f} KiB")
print(f"RPC reduction        = {reduction:,}x")

assert full_rpcs == 250_000
assert full_bytes == 10_000_000_000
assert bounded_rpcs == 5
assert bounded_bytes == 200_000
assert reduction == 50_000

### Try it 6. Choose a Bloom filter type from read patterns

**Task.** Write a function that returns the right Bloom filter setting for an access pattern: whole-row point gets, point gets of a few columns in a wide row, or range and prefix scans.

**Hint.** `ROW` filters on the row key, `ROWCOL` filters on row plus column, and Bloom filters do not help a scan of many rows, so `NONE` is correct there.

**Expected result.** The four patterns map to `ROW`, `ROWCOL`, `NONE`, and `NONE`.

In [ ]:
def choose_bloom(access_pattern):
    if access_pattern == "point get of the whole row":
        return "ROW"
    if access_pattern == "point get of a few columns in a wide row":
        return "ROWCOL"
    if access_pattern in ("range scan", "prefix scan"):
        return "NONE"
    raise ValueError(f"unknown access pattern: {access_pattern}")


patterns = [
    "point get of the whole row",
    "point get of a few columns in a wide row",
    "range scan",
    "prefix scan",
]
choices = {pattern: choose_bloom(pattern) for pattern in patterns}
for pattern, choice in choices.items():
    print(f"{pattern:<45} -> {choice}")

assert choices["point get of the whole row"] == "ROW"
assert choices["point get of a few columns in a wide row"] == "ROWCOL"
assert choices["range scan"] == "NONE"
assert choices["prefix scan"] == "NONE"

### Try it 7. Plan a bulk load

**Task.** A 2,000 GiB export compresses to a 0.4 ratio in HFiles. Plan a bulk load with a 10 GiB target HFile size and 100 pre-split regions: compute the stored size, the number of HFiles, and the reducer count.

**Hint.** Stored size is `input_gib * compression_ratio`. HFiles are `ceil(stored / target_hfile_gib)`, and set the number of reducers to the region count so each region receives one file per reducer.

**Expected result.** `800 GiB` stored, 80 HFiles, 100 reducers, and 100 target regions.

In [ ]:
import math

INPUT_GIB = 2_000
COMPRESSION_RATIO = 0.4
TARGET_HFILE_GIB = 10
PRE_SPLIT_REGIONS = 100

stored_gib = INPUT_GIB * COMPRESSION_RATIO
hfiles = math.ceil(stored_gib / TARGET_HFILE_GIB)
reducers = PRE_SPLIT_REGIONS
regions = PRE_SPLIT_REGIONS

print(f"stored size   = {stored_gib:.1f} GiB")
print(f"HFiles        = {hfiles} at about {TARGET_HFILE_GIB} GiB each")
print(f"reducers      = {reducers}")
print(f"target regions= {regions}")

assert stored_gib == 800.0
assert hfiles == 80
assert reducers == 100
assert regions == 100
assert hfiles / regions == 0.8